# Tool functions

In [64]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-6"

In [66]:
def _add_message(messages: list, role: str, text: str) -> None:
    message = {"role": role, "content": text}
    messages.append(message)

def add_user_message(messages: list, text: str) -> None:
    _add_message(messages, "user", text)

def add_assistant_message(messages: list, text: str) -> None:
    _add_message(messages, "assistant", text)

def chat(messages: list, system_prompt: str | None = None, temperature: float=1.0, stop: list | None = None) -> str:
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system_prompt is not None:
        params["system"] = system_prompt
    if stop is not None:
        params["stop_sequences"] = stop
    response = client.messages.create(
        **params
    )
    return response.content[0].text


In [48]:
# Tools and Schemas

from datetime import datetime, timedelta


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")
    pass

add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}


In [ ]:
def get_current_datetime(date_format: str = "%Y-%m-%d %H:%M:%S") -> str:
    if not date_format:
        raise ValueError("Date format cannot be empty")
    return datetime.now().strftime(date_format)

get_current_datetime("%H:%M")

'15:12'

# Tool schemas

In [10]:
from anthropic.types import ToolParam

get_current_datetime_schema = ToolParam({
  "name": "get_current_datetime",
  "description": "Returns the current date and time as a formatted string. Use this tool whenever the user asks for the current date, time, or datetime. The output format is controlled by a strftime-compatible format string (e.g., '%Y-%m-%d' for date only, '%H:%M:%S' for time only, '%A, %B %d %Y' for a human-readable date). Defaults to '%Y-%m-%d %H:%M:%S' if no format is provided. Do not call this tool if the user is asking about a historical or future date — it only reflects the system's current time.",
  "input_schema": {
    "type": "object",
    "properties": {
      "date_format": {
        "type": "string",
        "description": "A strftime-compatible format string that controls how the datetime is rendered. For example: '%Y-%m-%d %H:%M:%S' produces '2025-04-29 14:30:00', '%B %d, %Y' produces 'April 29, 2025', and '%I:%M %p' produces '02:30 PM'. Must not be an empty string. Defaults to '%Y-%m-%d %H:%M:%S'.",
        "default": "%Y-%m-%d %H:%M:%S"
      }
    },
    "required": []
  }
})

# Handling message blocks

In [12]:
messages = []

messages.append(
    {
        "role": "user",
        "content": "What the exact time, formatted in HH:MM:SS?" 
    }
)

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

response

Message(id='msg_01K5jGBkm8wmLc4ChMG7iM6J', container=None, content=[ToolUseBlock(id='toolu_017Vqjee8kUeBeUYTS4cZAFH', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=828, output_tokens=63, server_tool_use=None, service_tier='standard'))

In [19]:
messages.append({
    "role": "assistant",
    "content": response.content
})

In [23]:
result = get_current_datetime(**response.content[0].input)

In [24]:
messages.append({
    "role": "user",
    "content": [
        {"tool_use_id": response.content[0].id,
         "type": "tool_result",
         "content": get_current_datetime(**response.content[0].input),
         "is_error": False}
         ]
         }
)

In [25]:
response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

response

Message(id='msg_01N9Fvs3wMAJP3aNrikhUBEr', container=None, content=[TextBlock(citations=None, text='The exact time is **16:00:10** (4:00:10 PM).', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=908, output_tokens=23, server_tool_use=None, service_tier='standard'))

# Multi-turn conversations with tools

In [67]:
from anthropic.types import Message

def _add_message(messages: list, role: str, message) -> None:
    message = {"role": role, "content": message.content if isinstance(message, Message) else message}
    messages.append(message)

def add_user_message(messages: list, message) -> None:
    _add_message(messages, "user", message)

def add_assistant_message(messages: list, message) -> None:
    _add_message(messages, "assistant", message)

def chat(messages: list, system_prompt: str | None = None, temperature: float=1.0, stop: list | None = None, tools: list | None = None) -> Message:
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system_prompt is not None:
        params["system"] = system_prompt
    if stop is not None:
        params["stop_sequences"] = stop
    if tools is not None:
        params["tools"] = tools
    response = client.messages.create(
        **params
    )
    return response

In [27]:
messages = []

add_user_message(messages, "What the exact time, formatted in HH:MM:SS?")

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

response

Message(id='msg_01FcdquhkFymfVS4wX6jqq8V', container=None, content=[ToolUseBlock(id='toolu_01ENdHR8EUNKBDWYmqkMBjyA', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=828, output_tokens=63, server_tool_use=None, service_tier='standard'))

In [28]:
add_assistant_message(messages, response)

In [29]:
messages

[{'role': 'user', 'content': 'What the exact time, formatted in HH:MM:SS?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01ENdHR8EUNKBDWYmqkMBjyA', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]}]

In [ ]:
def text_from_message(message: Message) -> str:
    return "\n".join(
        [block.text for block in message.content if block.type == "text"]
    )

In [42]:
text_from_message(response)

''

In [36]:
messages[1].get('content')

[ToolUseBlock(id='toolu_01ENdHR8EUNKBDWYmqkMBjyA', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]

# Implementing multiple turns

In [44]:
import json

def run_tool(tool_name: str, tool_input: dict) -> str:
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    else:
        raise ValueError(f"Unknown tool: {tool_name}")

def run_tools(message: Message) -> list:
    tool_requests = [block for block in message.content if block.type == "tool_use"]

    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_blocks.append({
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False
            })
        except Exception as err:
            tool_result_blocks.append({
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {err}",
                "is_error": True
            })

    return tool_result_blocks

In [45]:
def run_conversation(messages):
    while True:
        response = chat(messages, tools=[get_current_datetime_schema])
        add_assistant_message(messages, response)
        print(text_from_message(response))
        
        if response.stop_reason != "tool_use":
            break
            
        tool_results = run_tools(response)
        add_user_message(messages, tool_results)
    
    return messages

In [46]:
messages = []

add_user_message(messages, "What the exact time, formatted in HH:MM:SS?  Also, what is the current time in SS format?")

run_conversation(messages)


The exact time in HH:MM:SS format is **18:35:55**.

The current time in SS (seconds) format is **55**.


[{'role': 'user',
  'content': 'What the exact time, formatted in HH:MM:SS?  Also, what is the current time in SS format?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01UgwK5y6bNuAM8yJ1Txa4PJ', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use'),
   ToolUseBlock(id='toolu_01NRZSXs4stQdtN8AbwzfeTC', caller=DirectCaller(type='direct'), input={'date_format': '%S'}, name='get_current_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01UgwK5y6bNuAM8yJ1Txa4PJ',
    'content': '"18:35:55"',
    'is_error': False},
   {'type': 'tool_result',
    'tool_use_id': 'toolu_01NRZSXs4stQdtN8AbwzfeTC',
    'content': '"55"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='The exact time in HH:MM:SS format is **18:35:55**.\n\nThe current time in SS (seconds) format is **55**.', type='text')]}]

# Using multiple tools

In [49]:
def run_tool(tool_name: str, tool_input: dict) -> str | None:
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)
    else:
        raise ValueError(f"Unknown tool: {tool_name}")

def run_conversation(messages):
    while True:
        response = chat(messages, tools=[get_current_datetime_schema, add_duration_to_datetime_schema, set_reminder_schema])
        add_assistant_message(messages, response)
        print(text_from_message(response))
        
        if response.stop_reason != "tool_use":
            break
            
        tool_results = run_tools(response)
        add_user_message(messages, tool_results)
    
    return messages

In [50]:
messages = []

add_user_message(messages, "Set a reminder for my doctors appointment.  It's 177 days after Jan 1, 2050.")

run_conversation(messages)

I'll help you set a reminder for your doctor's appointment. First, let me calculate the date that is 177 days after January 1, 2050.
Now I'll set a reminder for your doctor's appointment on June 27, 2050:
----
Setting the following reminder for 2050-06-27T12:00:00:
Doctor's appointment
----
Perfect! I've set a reminder for your doctor's appointment on **Monday, June 27, 2050** at 12:00 AM. You'll receive a notification at that time.


[{'role': 'user',
  'content': "Set a reminder for my doctors appointment.  It's 177 days after Jan 1, 2050."},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll help you set a reminder for your doctor's appointment. First, let me calculate the date that is 177 days after January 1, 2050.", type='text'),
   ToolUseBlock(id='toolu_01DgGdCpgV2tpoBP8hNFctnN', caller=DirectCaller(type='direct'), input={'datetime_str': '2050-01-01', 'duration': 177, 'unit': 'days', 'input_format': '%Y-%m-%d'}, name='add_duration_to_datetime', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01DgGdCpgV2tpoBP8hNFctnN',
    'content': '"Monday, June 27, 2050 12:00:00 AM"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="Now I'll set a reminder for your doctor's appointment on June 27, 2050:", type='text'),
   ToolUseBlock(id='toolu_01KkcxVrhHtdmfH3FJHsKcUK', caller=DirectCaller(type='dire

# Fine grained tool calling

In [52]:
# See `003_tool_streaming_completed.ipynb` for an example of how to stream tool calls, which is recommended for long-running tools like `set_reminder`.

# The text edit tool

In [53]:
# Implementation of the TextEditorTool
import os
import shutil
from typing import Optional, List


class TextEditorTool:
    def __init__(self, base_dir: str = "", backup_dir: str = ""):
        self.base_dir = base_dir or os.getcwd()
        self.backup_dir = backup_dir or os.path.join(self.base_dir, ".backups")
        os.makedirs(self.backup_dir, exist_ok=True)

    def _validate_path(self, file_path: str) -> str:
        abs_path = os.path.normpath(os.path.join(self.base_dir, file_path))
        if not abs_path.startswith(self.base_dir):
            raise ValueError(
                f"Access denied: Path '{file_path}' is outside the allowed directory"
            )
        return abs_path

    def _backup_file(self, file_path: str) -> str:
        if not os.path.exists(file_path):
            return ""
        file_name = os.path.basename(file_path)
        backup_path = os.path.join(
            self.backup_dir, f"{file_name}.{os.path.getmtime(file_path):.0f}"
        )
        shutil.copy2(file_path, backup_path)
        return backup_path

    def _restore_backup(self, file_path: str) -> str:
        file_name = os.path.basename(file_path)
        backups = [
            f for f in os.listdir(self.backup_dir) if f.startswith(file_name + ".")
        ]
        if not backups:
            raise FileNotFoundError(f"No backups found for {file_path}")

        latest_backup = sorted(backups, reverse=True)[0]
        backup_path = os.path.join(self.backup_dir, latest_backup)

        shutil.copy2(backup_path, file_path)
        return f"Successfully restored {file_path} from backup"

    def _count_matches(self, content: str, old_str: str) -> int:
        return content.count(old_str)

    def view(self, file_path: str, view_range: Optional[List[int]] = None) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if os.path.isdir(abs_path):
                try:
                    return "\n".join(os.listdir(abs_path))
                except PermissionError:
                    raise PermissionError(
                        "Permission denied. Cannot list directory contents."
                    )

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            with open(abs_path, "r", encoding="utf-8") as f:
                content = f.read()

            if view_range:
                start, end = view_range
                lines = content.split("\n")

                if end == -1:
                    end = len(lines)

                selected_lines = lines[start - 1 : end]

                result = []
                for i, line in enumerate(selected_lines, start):
                    result.append(f"{i}: {line}")

                return "\n".join(result)
            else:
                lines = content.split("\n")
                result = []
                for i, line in enumerate(lines, 1):
                    result.append(f"{i}: {line}")

                return "\n".join(result)

        except UnicodeDecodeError:
            raise UnicodeDecodeError(
                "utf-8",
                b"",
                0,
                1,
                "File contains non-text content and cannot be displayed.",
            )
        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot access file.")
        except Exception as e:
            raise type(e)(str(e))

    def str_replace(self, file_path: str, old_str: str, new_str: str) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            with open(abs_path, "r", encoding="utf-8") as f:
                content = f.read()

            match_count = self._count_matches(content, old_str)

            if match_count == 0:
                raise ValueError(
                    "No match found for replacement. Please check your text and try again."
                )
            elif match_count > 1:
                raise ValueError(
                    f"Found {match_count} matches for replacement text. Please provide more context to make a unique match."
                )

            # Create backup before modifying
            self._backup_file(abs_path)

            # Perform the replacement
            new_content = content.replace(old_str, new_str)

            with open(abs_path, "w", encoding="utf-8") as f:
                f.write(new_content)

            return "Successfully replaced text at exactly one location."

        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot modify file.")
        except Exception as e:
            raise type(e)(str(e))

    def create(self, file_path: str, file_text: str) -> str:
        try:
            abs_path = self._validate_path(file_path)

            # Check if file already exists
            if os.path.exists(abs_path):
                raise FileExistsError(
                    "File already exists. Use str_replace to modify it."
                )

            # Create parent directories if they don't exist
            os.makedirs(os.path.dirname(abs_path), exist_ok=True)

            # Create the file
            with open(abs_path, "w", encoding="utf-8") as f:
                f.write(file_text)

            return f"Successfully created {file_path}"

        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot create file.")
        except Exception as e:
            raise type(e)(str(e))

    def insert(self, file_path: str, insert_line: int, new_str: str) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            # Create backup before modifying
            self._backup_file(abs_path)

            with open(abs_path, "r", encoding="utf-8") as f:
                lines = f.readlines()

            # Handle line endings
            if lines and not lines[-1].endswith("\n"):
                new_str = "\n" + new_str

            # Insert at the beginning if insert_line is 0
            if insert_line == 0:
                lines.insert(0, new_str + "\n")
            # Insert after the specified line
            elif insert_line > 0 and insert_line <= len(lines):
                lines.insert(insert_line, new_str + "\n")
            else:
                raise IndexError(
                    f"Line number {insert_line} is out of range. File has {len(lines)} lines."
                )

            with open(abs_path, "w", encoding="utf-8") as f:
                f.writelines(lines)

            return f"Successfully inserted text after line {insert_line}"

        except ValueError as e:
            raise ValueError(str(e))
        except PermissionError:
            raise PermissionError("Permission denied. Cannot modify file.")
        except Exception as e:
            raise type(e)(str(e))

    def undo_edit(self, file_path: str) -> str:
        try:
            abs_path = self._validate_path(file_path)

            if not os.path.exists(abs_path):
                raise FileNotFoundError("File not found")

            return self._restore_backup(abs_path)

        except ValueError as e:
            raise ValueError(str(e))
        except FileNotFoundError:
            raise FileNotFoundError("No previous edits to undo")
        except PermissionError:
            raise PermissionError("Permission denied. Cannot restore file.")
        except Exception as e:
            raise type(e)(str(e))

In [ ]:
# Process Tool Call Requests
import json

text_editor_tool = TextEditorTool()

# Edited to change tool name from `str_replace_editor` to `str_replace_based_edit_tool`
def run_tool(tool_name, tool_input):
    if tool_name == "str_replace_based_edit_tool":
        command = tool_input["command"]
        if command == "view":
            return text_editor_tool.view(
                tool_input["path"], tool_input.get("view_range")
            )
        elif command == "str_replace":
            return text_editor_tool.str_replace(
                tool_input["path"], tool_input["old_str"], tool_input["new_str"]
            )
        elif command == "create":
            return text_editor_tool.create(tool_input["path"], tool_input["file_text"])
        elif command == "insert":
            return text_editor_tool.insert(
                tool_input["path"],
                tool_input["insert_line"],
                tool_input["new_str"],
            )
        elif command == "undo_edit":
            return text_editor_tool.undo_edit(tool_input["path"])
        else:
            raise Exception(f"Unknown text editor command: {command}")
    else:
        raise Exception(f"Unknown tool name: {tool_name}")


def run_tools(message):
    tool_requests = [block for block in message.content if block.type == "tool_use"]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False,
            }
        except Exception as e:
            tool_result_block = {
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True,
            }

        tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

In [ ]:
# Make the text edit schema based on the model version being used
def get_text_edit_schema(model):
    return {
        "type": "text_editor_20250728",
        "name": "str_replace_based_edit_tool",
    }

In [56]:
# Run the conversation in a loop until the model doesn't ask for a tool use
def run_conversation(messages):
    while True:
        response = chat(
            messages,
            tools=[get_text_edit_schema(model)],
        )

        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages

In [59]:
messages = []

add_user_message(
    messages,
    """
    Open the ./artifacts/main.py file and summarize its contents.
    """,
)

run_conversation(messages)


Let me try with the current directory:
## Summary of ./artifacts/main.py

The file contains a simple Python script with one function:

- **Function: `greeting()`** - A basic function that prints the string "Hello, world!" to the console when called.

This is a minimal, beginner-level Python script that demonstrates the basic concept of defining and using a function.


[{'role': 'user',
  'content': '\n    Open the ./artifacts/main.py file and summarize its contents.\n    '},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_01K9SD5mf6yXJseuv4FJ7kap', caller=DirectCaller(type='direct'), input={'command': 'view', 'path': '/artifacts'}, name='str_replace_based_edit_tool', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01K9SD5mf6yXJseuv4FJ7kap',
    'content': "Error: Access denied: Path '/artifacts' is outside the allowed directory",
    'is_error': True}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='Let me try with the current directory:', type='text'),
   ToolUseBlock(id='toolu_01ChimBte5vLEsh5VrvyWcVW', caller=DirectCaller(type='direct'), input={'command': 'view', 'path': './artifacts/main.py'}, name='str_replace_based_edit_tool', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01ChimBte5vLEsh5VrvyWcVW',
  

In [60]:
messages = []

add_user_message(
    messages,
    """
    Open the ./artifacts/main.py file and add a function to calculate pi to the 5th digit.

    Then, create a ./artifacts/test.py file to test your implementation.
    """,
)

run_conversation(messages)

I'll start by examining the current main.py file, then add a function to calculate pi to the 5th digit, and create a test file.
Let me try viewing the file with the correct path:
Great! Now I'll add a function to calculate pi to the 5th digit. I'll use the Machin formula which converges quickly:
Now, let's create the test file:
Perfect! Let me verify the updated main.py file:
Excellent! I've successfully completed your requests:

## Summary

### 1. **Updated `./artifacts/main.py`**
   - Added a new function `calculate_pi_fifth_digit()` that calculates pi to the 5th decimal place (3.14159)
   - The implementation uses **Machin's formula**: `π/4 = 4×arctan(1/5) - arctan(1/239)`
   - Uses the `Decimal` module for high precision arithmetic
   - Uses Taylor series to calculate the arctangent values
   - Returns the result as a float rounded to 5 decimal places

### 2. **Created `./artifacts/test.py`**
   - Contains comprehensive unit tests using Python's `unittest` framework
   - **Test cas

[{'role': 'user',
  'content': '\n    Open the ./artifacts/main.py file and add a function to calculate pi to the 5th digit.\n\n    Then, create a ./artifacts/test.py file to test your implementation.\n    '},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll start by examining the current main.py file, then add a function to calculate pi to the 5th digit, and create a test file.", type='text'),
   ToolUseBlock(id='toolu_0128x6nngfwXHo4X3HhqEaNW', caller=DirectCaller(type='direct'), input={'command': 'view', 'path': '/artifacts'}, name='str_replace_based_edit_tool', type='tool_use')]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_0128x6nngfwXHo4X3HhqEaNW',
    'content': "Error: Access denied: Path '/artifacts' is outside the allowed directory",
    'is_error': True}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='Let me try viewing the file with the correct path:', type='text'),
   ToolUseBlock(id='to

# The web search tool

In [62]:
web_search_schema = {
    "type": "web_search_20260209",
    "name": "web_search",
    "max_uses": 5,
}

In [68]:
messages = []

add_user_message(
    messages,
    """
    What's the best exercise for gaining leg muscle?
    """,
)

response = chat(
    messages,
    tools=[web_search_schema]
)  
response

Message(id='msg_012gXRXbZdpHfzwuq5NXJ7AY', container=Container(id='container_011CaYw9PmNJSq1dedaKVXHQ', expires_at=datetime.datetime(2026, 4, 30, 0, 31, 50, 387077, tzinfo=TzInfo(0))), content=[ServerToolUseBlock(id='srvtoolu_01K7VPcmhDxmAqE56yRXGzgr', caller=None, input={'code': '\nresults = await web_search({"query": "best exercises for gaining leg muscle"})\nimport json\nfor i, r in enumerate(results):\n    print(f"Result {i}: {r[\'title\']}")\n    print(r[\'content\'][:300])\n    print()\n'}, name='code_execution', type='server_tool_use'), ServerToolUseBlock(id='srvtoolu_01LHzUmchS7S1ocvEZvMw2Dd', caller=ServerToolCaller20260120(tool_id='srvtoolu_01K7VPcmhDxmAqE56yRXGzgr', type='code_execution_20260120'), input={'query': 'best exercises for gaining leg muscle'}, name='web_search', type='server_tool_use'), WebSearchToolResultBlock(caller=ServerToolCaller20260120(tool_id='srvtoolu_01K7VPcmhDxmAqE56yRXGzgr', type='code_execution_20260120'), content=[WebSearchResultBlock(encrypted_cont

In [69]:
web_search_schema = {
    "type": "web_search_20260209",
    "name": "web_search",
    "max_uses": 5,
    "allowed_domains": ["nih.gov"]
}

In [70]:
messages = []

add_user_message(
    messages,
    """
    What's the best exercise for gaining leg muscle?
    """,
)

response = chat(
    messages,
    tools=[web_search_schema]
)  
response

Message(id='msg_01GEZLzJtsdZ7vJacH8EeMYr', container=Container(id='container_011CaYwTmttBT81rFqKWyWd1', expires_at=datetime.datetime(2026, 4, 30, 0, 36, 25, 243261, tzinfo=TzInfo(0))), content=[ServerToolUseBlock(id='srvtoolu_01FnZWo6yfa34u2ZsCcjo8sB', caller=None, input={'code': '\nresults = await web_search({"query": "best exercises for gaining leg muscle"})\nimport json\nfor i, r in enumerate(results):\n    print(f"Result {i}: {r[\'title\']}")\n    print(r[\'content\'][:300])\n    print("---")\n'}, name='code_execution', type='server_tool_use'), ServerToolUseBlock(id='srvtoolu_01J5FM3qhPwJcnW1Cg2Zb5Jh', caller=ServerToolCaller20260120(tool_id='srvtoolu_01FnZWo6yfa34u2ZsCcjo8sB', type='code_execution_20260120'), input={'query': 'best exercises for gaining leg muscle'}, name='web_search', type='server_tool_use'), WebSearchToolResultBlock(caller=ServerToolCaller20260120(tool_id='srvtoolu_01FnZWo6yfa34u2ZsCcjo8sB', type='code_execution_20260120'), content=[WebSearchResultBlock(encrypted